# Step 4: Mendelian Randomization
Two-sample MR using eQTLGen blood cis-eQTL data as the exposure and T2D GWAS as the outcome.
For each gene, genome-wide significant eQTL instruments are retrieved via the OpenGWAS `/tophits` endpoint,
then tested against T2D associations using Wald ratio (single SNP) or IVW (multiple SNPs).

In [ ]:
from gwas_explorer.mr_analysis import run_mr_analysis
from gwas_explorer.config import PROCESSED_DIR
import pandas as pd

candidate_genes = pd.read_parquet(PROCESSED_DIR / "t2d_candidate_genes.parquet")
mr_results = run_mr_analysis(candidate_genes)
status_counts = mr_results["MR_STATUS"].value_counts()
print(f"MR results:\n{status_counts}")
mr_results.head(10)

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

ok = mr_results[mr_results["MR_STATUS"] == "ok"]
if len(ok) > 0:
    fig, ax = plt.subplots(figsize=(8, 6))
    ax.scatter(ok["MR_ESTIMATE"], -np.log10(ok["MR_PVALUE"]), alpha=0.6)
    ax.axhline(-np.log10(0.05), color="red", linestyle="--", label="p = 0.05")
    ax.set_xlabel("MR Estimate (causal effect)")
    ax.set_ylabel("-log10(p-value)")
    ax.set_title("Mendelian Randomization Results")
    ax.legend()
    plt.tight_layout()
    plt.show()
else:
    print("No MR results with status 'ok' to plot.")

In [ ]:
mr_results.to_parquet(PROCESSED_DIR / "t2d_mr_results.parquet", index=False)
print(f"Saved MR results for {len(mr_results)} genes")